<a href="https://colab.research.google.com/github/anumit2004/Attention-free-Transformer/blob/main/aft_simple.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## AFT-SIMPLE with Stabilised key

In [ ]:
import torch
import math
from torch import nn
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from datasets import load_dataset
from transformers import AutoTokenizer

In [ ]:
import torch
import torch.nn as nn

# =============================================================================
# AFT-SIMPLE MODULE
# =============================================================================
class AFTSimple(nn.Module):
    def __init__(self, seq_len, dim):
        super().__init__()
        self.dim = dim
        self.seq_len = seq_len

        self.q_proj = nn.Linear(dim, dim)
        self.k_proj = nn.Linear(dim, dim)
        self.v_proj = nn.Linear(dim, dim)

        self.out_proj = nn.Linear(dim, dim)

    def forward(self, x):
        B, T, d = x.shape

        Q = self.q_proj(x)
        K = self.k_proj(x)
        V = self.v_proj(x)

        Q_sig = torch.sigmoid(Q)

        # Numerical stability against exponential overflow
        K_stable = K - K.max(dim=1, keepdim=True)[0]
        exp_K = torch.exp(K_stable)

        # AFT-Simple relies on a global sum over the sequence dimension (dim=1)
        # We no longer calculate 'w' or matrix multiply a local mask
        numerator = torch.sum(exp_K * V, dim=1, keepdim=True)
        denominator = torch.sum(exp_K, dim=1, keepdim=True) + 1e-6

        # Calculate final context and gate it with Query
        context = numerator / denominator
        Y = Q_sig * context

        return self.out_proj(Y)

In [ ]:
class MLP(nn.Module):
    def __init__(self, dim, hidden_dim, dp=0.1):
        super().__init__()
        self.l1 = nn.Linear(dim, hidden_dim)
        self.g1 = nn.GELU()
        self.l2 = nn.Linear(hidden_dim, dim)
        self.d1 = nn.Dropout(dp)

    def forward(self, x):
        x = self.l1(x)
        x = self.g1(x)
        x = self.d1(x)
        return self.l2(x)

In [ ]:
class AFTEncoderBlock(nn.Module):
    def __init__(self, max_seqlen, dim, hidden_dim, p=0.1):
        super().__init__()
        self.ln1 = nn.LayerNorm(dim)
        self.ln2 = nn.LayerNorm(dim)

        self.attn = AFTSimple(seq_len=max_seqlen, dim=dim)

        self.mlp = MLP(dim, hidden_dim, dp=p)
        self.d1 = nn.Dropout(p)
        self.d2 = nn.Dropout(p)

    def forward(self, x):
        x_norm = self.ln1(x)
        x = x + self.d1(self.attn(x_norm))
        x_norm = self.ln2(x)
        out = x + self.d2(self.mlp(x_norm))
        return out

In [ ]:
class AFT(nn.Module):
    def __init__(self, vocab_size, max_seqlen, dim, hidden_dim, depth=4, p=0.1):
        super().__init__()
        self.dim = dim
        self.embed = nn.Embedding(vocab_size, dim)

        # Simple learnable absolute positional embeddings
        self.pos_embed = nn.Embedding(max_seqlen, dim)

        self.enc = nn.Sequential(*[
            AFTEncoderBlock(max_seqlen, dim, hidden_dim, p=p)
            for _ in range(depth)
        ])
        self.dec = nn.Linear(dim, vocab_size)

    def forward(self, x):
        B, T = x.shape
        device = x.device

        positions = torch.arange(0, T, device=device).unsqueeze(0).expand(B, T)
        x = self.embed(x) * math.sqrt(self.dim) + self.pos_embed(positions)

        x = self.enc(x)
        out = self.dec(x)
        return out


In [ ]:
class OpusTextDataset(Dataset):
    """A clean dataset class that just holds pre-processed token chunks."""
    def __init__(self, input_ids_list):
        self.input_ids = input_ids_list

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        chunk = torch.tensor(self.input_ids[idx], dtype=torch.long)
        x = chunk[:-1]
        y = chunk[1:]
        return x, y

In [ ]:
from torch.utils.data import Dataset, DataLoader, random_split
def prepare_dataloaders(max_seqlen=64, batch_size=16, max_samples=5000, tokenizer_name="gpt2"):
    print("Loading OPUS Books dataset via Hugging Face...")
    raw_dataset = load_dataset("Helsinki-NLP/opus_books", "en-fr", split="train")

    print("Initializing Tokenizer...")
    tokenizer = AutoTokenizer.from_pretrained(tokenizer_name)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    print("Tokenizing and chunking text corpus...")
    buffer = []
    all_input_ids = []

    for item in raw_dataset:
        text = item['translation']['en']
        tokens = tokenizer.encode(text)
        buffer.extend(tokens)

        while len(buffer) >= (max_seqlen + 1):
            all_input_ids.append(buffer[:max_seqlen + 1])
            buffer = buffer[max_seqlen:]


        if max_samples != None :
            if len(all_input_ids) >= max_samples:
                all_input_ids = all_input_ids[:max_samples]
                break

    # 1. Wrap all chunks in our dataset class
    full_dataset = OpusTextDataset(all_input_ids)

    # 2. Calculate split sizes (80% Train, 10% Validation, 10% Test)
    total_size = len(full_dataset)
    train_size = int(0.8 * total_size)
    val_size = int(0.1 * total_size)
    test_size = total_size - train_size - val_size

    # 3. Perform the random split
    generator = torch.Generator().manual_seed(42) # Seed for reproducibility
    train_data, val_data, test_data = random_split(
        full_dataset,
        [train_size, val_size, test_size],
        generator=generator
    )

    print(f"Data Split Complete: {len(train_data)} Train | {len(val_data)} Val | {len(test_data)} Test")

    # 4. Create DataLoaders
    train_loader = DataLoader(train_data, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_data, batch_size=batch_size, shuffle=False)
    test_loader = DataLoader(test_data, batch_size=batch_size, shuffle=False)

    return train_loader, val_loader, test_loader, tokenizer

In [ ]:
import torch.nn as nn

# =============================================================================
# 3. TRAINING ROUTINE WITH VALIDATION
# =============================================================================

def train_on_opus():
    # Hyperparameters
    MAX_SEQLEN = 64
    EMBED_DIM = 512
    HIDDEN_DIM = 512
    DEPTH = 10
    BATCH_SIZE = 16
    EPOCHS = 5
    LR = 5e-4
    DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    # Fetch our newly split dataloaders
    train_loader, val_loader, test_loader, tokenizer = prepare_dataloaders(
        max_seqlen=MAX_SEQLEN,
        batch_size=BATCH_SIZE,
        # max_samples=None
    )
    VOCAB_SIZE = tokenizer.vocab_size

    print(f"\nConfiguration Details:")
    print(f"-> Device: {DEVICE}")
    print(f"-> Model Vocabulary Size: {VOCAB_SIZE}")
    print("Initializing AFT Model Architecture...")

    # Build Model
    model = AFT(
        vocab_size=VOCAB_SIZE,
        max_seqlen=MAX_SEQLEN,
        dim=EMBED_DIM,
        hidden_dim=HIDDEN_DIM,
        depth=DEPTH
    ).to(DEVICE)

    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
    criterion = nn.CrossEntropyLoss()

    print("\nBeginning Training Pipeline...")

    for epoch in range(EPOCHS):
        # --- TRAINING PHASE ---
        model.train()
        train_loss = 0.0

        for batch_idx, (inputs, targets) in enumerate(train_loader):
            inputs, targets = inputs.to(DEVICE), targets.to(DEVICE)

            optimizer.zero_grad()
            outputs = model(inputs)

            loss = criterion(outputs.view(-1, VOCAB_SIZE), targets.view(-1))
            loss.backward()

            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

            train_loss += loss.item()

            if batch_idx % 20 == 0:
                print(f"Epoch {epoch+1}/{EPOCHS} | Train Batch {batch_idx}/{len(train_loader)} | Loss: {loss.item():.4f}")

        avg_train_loss = train_loss / len(train_loader)

        # --- VALIDATION PHASE ---
        model.eval()
        val_loss = 0.0
        correct_tokens = 0
        total_tokens = 0

        with torch.no_grad():
            for inputs, targets in val_loader:
                inputs, targets = inputs.to(DEVICE), targets.to(DEVICE)
                outputs = model(inputs)

                loss = criterion(outputs.view(-1, VOCAB_SIZE), targets.view(-1))
                val_loss += loss.item()

                preds = outputs.argmax(dim=-1)
                correct_tokens += (preds == targets).sum().item()
                total_tokens += targets.numel()

        avg_val_loss = val_loss / len(val_loader)
        val_accuracy = (correct_tokens / total_tokens) * 100

        print(f"=== Epoch {epoch+1:02d} Complete ===")
        print(f"Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f} | Val Accuracy: {val_accuracy:.2f}%\n")

    # Return the test_loader as well so you can run final evaluations later
    return model, tokenizer, test_loader

if __name__ == '__main__':
    model, tokenizer, test_loader = train_on_opus()

In [ ]:

# =============================================================================
# FINAL TEST SET EVALUATION
# =============================================================================

def evaluate_test_set(model, test_loader, tokenizer, device):
    print("\n--- Running Final Evaluation on Unseen Test Set ---")
    model.eval()
    criterion = nn.CrossEntropyLoss()
    VOCAB_SIZE = tokenizer.vocab_size

    test_loss = 0.0
    correct_tokens = 0
    total_tokens = 0

    with torch.no_grad():
        for batch_idx, (inputs, targets) in enumerate(test_loader):
            inputs, targets = inputs.to(device), targets.to(device)

            # Forward pass
            outputs = model(inputs)

            # Calculate loss
            loss = criterion(outputs.view(-1, VOCAB_SIZE), targets.view(-1))
            test_loss += loss.item()

            # Calculate accuracy
            preds = outputs.argmax(dim=-1)
            correct_tokens += (preds == targets).sum().item()
            total_tokens += targets.numel()

    avg_test_loss = test_loss / len(test_loader)
    test_accuracy = (correct_tokens / total_tokens) * 100

    print(f"Final Test Loss: {avg_test_loss:.4f}")
    print(f"Final Test Accuracy: {test_accuracy:.2f}%")
    print("---------------------------------------------------\n")

# Set the device
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Run the evaluation using the variables returned from train_on_opus()
evaluate_test_set(model, test_loader, tokenizer, DEVICE)

In [ ]:
# 1. Define the generation function (with temperature for better text)
def generate_text(model, tokenizer, prompt, max_new_tokens=50, device='cpu', temperature=0.8):
    model.eval() # Set the model to evaluation mode

    # Safety check: Ensure the model knows its max_seqlen (since it wasn't in AFT __init__)
    if not hasattr(model, 'max_seqlen'):
        model.max_seqlen = 64

    encoded_prompt = tokenizer.encode(prompt, return_tensors='pt').to(device)
    generated_sequence = encoded_prompt.tolist()[0]

    print(f"\nGenerating text with prompt: '{prompt}'")

    with torch.no_grad():
        for _ in range(max_new_tokens):
            # Take only the last MAX_SEQLEN tokens if the sequence exceeds it
            current_input = torch.tensor([generated_sequence[-model.max_seqlen:]], dtype=torch.long).to(device)

            # Get predictions for the next token
            outputs = model(current_input)
            next_token_logits = outputs[0, -1, :] / temperature # Apply temperature

            # Sample the next token using multinomial distribution with temperature
            probs = torch.softmax(next_token_logits, dim=-1)
            next_token = torch.multinomial(probs, num_samples=1).item()

            generated_sequence.append(next_token)

            # Stop if EOS token is generated
            if next_token == tokenizer.eos_token_id:
                break

    return tokenizer.decode(generated_sequence)

# 2. Set the device
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# 3. Test the model! (Using 'model' and 'tokenizer' already in your environment)
prompt = "The quick brown fox"
generated_text = generate_text(model, tokenizer, prompt, max_new_tokens=50, device=DEVICE)
print("\nGenerated Text 1:")
print(generated_text)

prompt_2 = "Once upon a time, in a land far, far away"
generated_text_2 = generate_text(model, tokenizer, prompt_2, max_new_tokens=50, device=DEVICE)
print("\nGenerated Text 2:")
print(generated_text_2)